In [1]:
from transformers import AutoTokenizer, BertForSequenceClassification

# bert-base-uncased
# prajjwal1/bert-tiny

model_name = "prajjwal1/bert-tiny"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2, 
    id2label={
        0: "Negative",
        1: "Positive"
    }
)

base_model.bert

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-1): 2 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=128, out_features=128, bias=True)
            (key): Linear(in_features=128, out_features=128, bias=True)
            (value): Linear(in_features=128, out_features=128, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=128, out_features=128, bias=True)
            (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [2]:
from datasets import load_dataset

def tokenize_function(examples):
    return tokenizer(examples["text"], max_length=128, padding="max_length", truncation=True)

dataset = load_dataset("fancyzhx/yelp_polarity")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [3]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [4]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [5]:
import torch
import torch.nn as nn
import copy


class ReSpropLinear(nn.Linear):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        device=None,
        dtype=None,
        reuse_percentage: float = 0.9
    ):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.prev_gradients = None
        self.reuse_percentage = reuse_percentage
        
    def forward(self, *args, **kwargs):
        output = super().forward(*args, **kwargs)
    
        if self.training:
            def custom_backward(grad_output):
                current_gradients = grad_output.clone()
                
                if self.prev_gradients is not None:
                    # Calculate gradient difference
                    grad_diff = torch.abs(current_gradients - self.prev_gradients)
                    
                    # Find threshold
                    if grad_diff.device.type == "mps":
                        sorted_diffs = torch.sort(grad_diff.flatten())[0]
                        threshold_idx = int(len(sorted_diffs) * self.reuse_percentage)
                        threshold = sorted_diffs[threshold_idx]
                    else:
                        threshold_idx = int(len(grad_diff.flatten()) * self.reuse_percentage)
                        threshold = torch.kthvalue(grad_diff.flatten(), threshold_idx)[0]
                    
                    # Create mask for gradients to reuse
                    reuse_mask = grad_diff <= threshold
                    
                    # Combine current and previous gradients
                    hybrid_gradients = torch.where(
                        reuse_mask,
                        self.prev_gradients,
                        current_gradients
                    )
                else:
                    hybrid_gradients = current_gradients
                
                # Store gradients for next iteration
                # There are many ways of doing this:
                # - Store all gradients for the mini batch
                # - Average the gradient
                # - Random sampling
                self.prev_gradients = torch.mean(current_gradients, dim=0).detach()
                
                return hybrid_gradients
            
            if output.requires_grad:
                output.register_hook(custom_backward)
        
        return output
    
    def extra_repr(self):
        return super().extra_repr() + f", reuse_percentage={self.reuse_percentage}"
    
def resprofify_bert(base_model, reuse_percentage=0.9):
    def resprop_linear(layer: nn.Linear):
        return ReSpropLinear(
            layer.in_features, 
            layer.out_features, 
            layer.bias is not None,
            reuse_percentage=reuse_percentage
        )
        
    model = copy.deepcopy(base_model)
    for layer in model.bert.encoder.layer:
        # Self Attention
        att = layer.attention
        att.self.query   = resprop_linear(att.self.query)
        att.self.key     = resprop_linear(att.self.key)
        att.self.value   = resprop_linear(att.self.value)
        att.output.dense = resprop_linear(att.output.dense)
        
        # Feed Forward Block
        layer.intermediate.dense = resprop_linear(layer.intermediate.dense)
        layer.output.dense       = resprop_linear(layer.output.dense)
        
    return model


In [6]:
model = resprofify_bert(base_model, reuse_percentage=0.99)

# Freeze Bert
# model.bert.requires_grad_(False)

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="trainer_out", eval_strategy="epoch")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)


In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.676700,0.539186,0.757000
2,0.543100,0.447956,0.801000
3,0.495700,0.434173,0.805000


TrainOutput(global_step=1875, training_loss=0.54951630859375, metrics={'train_runtime': 95.2026, 'train_samples_per_second': 157.559, 'train_steps_per_second': 19.695, 'total_flos': 4764326400000.0, 'train_loss': 0.54951630859375, 'epoch': 3.0})

In [9]:
inputs = tokenizer("love this place!", return_tensors="pt")

with torch.no_grad():
    logits = model(
        **{k : v.to(model.device) for k, v in inputs.items() }
    ).logits

predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

'Positive'

In [10]:
trainer.evaluate()

{'eval_loss': 0.4341731369495392,
 'eval_accuracy': 0.805,
 'eval_runtime': 1.5306,
 'eval_samples_per_second': 653.345,
 'eval_steps_per_second': 81.668,
 'epoch': 3.0}